# Notebook 00: Post-Training 方法论概述

本 Notebook 讲解大模型 Post-training 的核心概念，涵盖 Tülu 3 论文的关键方法论。

**目标读者**：了解 Pre-training 基本概念，想深入理解 Post-training 全链路的工程师。

---

## 概念 1：为什么 Base Model 只会"接龙"不会对话

### Base Model 的训练目标

Base Model（如 Llama-3.1-8B、Qwen2.5-1.5B）的训练目标只有一个：

$$\mathcal{L} = -\sum_{t} \log P(x_t | x_{<t})$$

即给定前文，预测下一个 token。这叫 **Causal Language Modeling**（因果语言建模）。

### 为什么不会对话？

训练数据是互联网文本的海洋：网页、书籍、论文、论坛帖子、代码...

模型学到的是"互联网上的文字长什么样"，而不是"我应该扮演助手来回答问题"。

**示例：给 Base Model 输入"法国的首都是什么？"**

它可能续写出：
- `「法国的首都是什么？」这道题出现在2023年高考地理卷中...`（题库格式）
- `法国的首都是什么？\n\nA. 巴黎 B. 伦敦 C. 柏林 D. 马德里`（选择题格式）
- `法国的首都是什么？我在知乎上看到有人问这个问题...`（论坛格式）
- `法国的首都是巴黎。德国的首都是柏林。意大利的...`（百科列表格式）

每种续写都是"合理的下一句话"，但没有一种是"以助手身份回答你的问题"。

### 关键洞察

> **Base Model 不知道自己应该扮演"助手"角色。** 它只是一个强大的文本续写器。
> Post-training 的第一步（SFT）就是教它"你是助手，用户问你问题，你要回答"。

In [ ]:
# 演示：Base Model 的续写行为
# 如果 Qwen2.5-1.5B Base Model 可用，实际观察它的输出

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # Base Model（不是 Instruct 版本）

# 加载模型
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, trust_remote_code=True
    )
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = model.to(device)
    print(f"模型加载成功: {MODEL_NAME}, 设备: {device}")
except Exception as e:
    print(f"模型加载失败: {e}")
    print("跳过演示，请先下载模型")

In [ ]:
# 观察 Base Model 对"问题"的续写——它不会"回答"，只会"接龙"

test_prompts = [
    "What is the capital of France?",
    "Explain photosynthesis in simple terms.",
    "How do I write a Python function to sort a list?",
]

print("=" * 60)
print("Base Model 续写演示")
print("=" * 60)

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=100, do_sample=True,
            temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n--- Prompt: {prompt} ---")
    print(f"续写: {response}")
    print()

print("\n观察: Base Model 不是在'回答问题'，而是在'续写文本'")
print("它可能续写成题库、论坛帖子、百科等各种格式")

## 概念 2：SFT 的本质——学会对话格式

### 什么是 SFT（Supervised Fine-Tuning）？

SFT 用"用户问 → 助手答"格式的数据继续训练 Base Model：

```
<|user|>
法国的首都是什么？
<|assistant|>
法国的首都是巴黎。巴黎位于法国北部塞纳河畔，是法国最大的城市...
```

训练几万条这样的数据后，模型就学会了：
1. 看到 `<|user|>` 标记 → 这是用户在问我问题
2. 在 `<|assistant|>` 之后 → 我应该以助手身份回答
3. 回答的风格 → 应该是有帮助的、直接的、结构化的

### SFT 学到了什么？

| 学到的能力 | 没有学好的 |
|-----------|----------|
| 对话格式（知道要回答问题） | 回答质量（可能啰嗦、不准确） |
| 角色意识（我是助手） | 安全意识（可能回答有害问题） |
| 基本礼貌和结构 | 偏好对齐（不知道什么是"好"回复） |

### Tülu 3 的 SFT 数据配比

**Tülu 3 不是把所有数据扔到一起训练**，而是精心调配每种技能的数据比例：

- 通用指令（FLAN v2）：10K → 教模型理解各种指令格式
- 数学推理（NuminaMath）：5K → 教模型数学解题
- 安全拒绝（WildGuardMix）：10K → 教模型拒绝有害请求
- 防过度拒绝（CoCoNot）：5K → 教模型不误拒良性问题
- 人工高质量（No Robots）：9.5K → 人工标注的高质量对话
- ... 总计 ~57K 条

> **对应 Tülu 3 论文 Section 3**：数据量不是越多越好。过多的安全数据导致 over-refusal，过多的合成数据导致风格单一。
> 关键是找到平衡点——"技能隔离"方法帮助找到每个技能的最优数据量。

## 概念 3：DPO 的本质——学会选更好的回复

### SFT 之后的问题

SFT 后的模型会对话了，但回答质量参差不齐。同一个问题，它可能生成：
- 好回复：简洁准确，直接回答
- 差回复：啰嗦、跑题、甚至有害

**怎么让模型学会"生成好回复，避免差回复"？**

### DPO（Direct Preference Optimization）

DPO 的训练数据是**偏好对**（preference pairs）：

```
Prompt:    "法国的首都是什么？"
Chosen:    "法国的首都是巴黎。"                    ← 人类偏好的好回复
Rejected:  "我不太确定，可能是巴黎或者里昂..."      ← 人类不偏好的差回复
```

### DPO 不是做选择题

一个常见误解是"DPO 让模型做选择题"。实际上：

DPO **调整模型参数**，使得：
- 生成 chosen 类型回复的概率 ↑
- 生成 rejected 类型回复的概率 ↓

训练完成后，面对**新问题**（训练时没见过的），模型自然倾向于生成更好的回复。

### DPO 的数学直觉

DPO loss 函数：

$$\mathcal{L}_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \left(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)\right]$$

其中：
- $\pi_\theta$：正在训练的模型
- $\pi_{ref}$：参考模型（SFT 模型的冻结副本）
- $y_w$：chosen 回复
- $y_l$：rejected 回复
- $\beta$：控制偏离 SFT 的程度

**直觉**：让 chosen 相对于 ref 的概率比 rejected 相对于 ref 的概率更高。

### DPO vs RLHF

| | RLHF | DPO |
|--|------|-----|
| 步骤 | SFT → 训练 RM → PPO 优化（三步） | SFT → DPO 优化（两步） |
| 需要 RM？ | 是，需要单独训练 Reward Model | 否，直接用偏好对 |
| 稳定性 | PPO 不稳定，超参敏感 | 更稳定，更容易调参 |
| 理论等价？ | — | 在 Bradley-Terry 假设下等价 |
| 局限 | RM 训练成本高 | 假设偏好符合 BT 模型，可能过于简化 |

> **对应 Tülu 3 论文 Section 4**：Tülu 3 选择 DPO 而非 RLHF，因为更简单、更稳定，且效果相当。

## 概念 4：DPO 数据构建方法

### 偏好对从哪来？

Tülu 3 的 DPO 数据构建流程：

```
Prompt → 多模型生成候选回复 → GPT-4o 四维度评分 → 最高分=Chosen, 较低分=Rejected
```

**四个评分维度**：
1. **Helpfulness（有帮助性）**：回答是否解决了用户的问题
2. **Correctness（正确性）**：回答内容是否事实正确
3. **Coherence（连贯性）**：回答是否逻辑连贯、结构清晰
4. **Safety（安全性）**：回答是否安全、没有有害内容

### On-policy vs Off-policy

| 类型 | 回复生成者 | 特点 |
|------|----------|------|
| On-policy | 训练中的模型自己生成 | 学"纠正自己的错误" |
| Off-policy | 其他模型生成 | 学"避免别人的错误" |

Tülu 3 发现**混合使用效果最好**：
- On-policy 让模型看到"自己可能犯的错"→ 更有效地自我改进
- Off-policy 提供更多样的错误模式 → 泛化能力更强

> **对应 Tülu 3 论文 Section 4.1**：On-policy 数据占比约 30%，其余为 Off-policy。
> 纯 Off-policy 效果不如混合使用。

## 概念 5：RLVR 原理（本项目不实现，只讲解）

### 什么是 RLVR？

**RLVR = Reinforcement Learning with Verifiable Rewards**

对有"标准答案"的任务，直接用答案对错作为奖励信号，不需要训练 Reward Model。

### 适用场景

| 任务类型 | 验证方法 | 示例 |
|---------|---------|------|
| 数学 | 代入检查、数值对比 | 答案是 42 → 对/错 |
| 代码 | 执行测试用例 | 通过所有测试 → 对/错 |
| 指令遵循 | 程序化检查约束 | "恰好 5 条" → 数条数 |
| 格式约束 | 正则匹配 | "用 JSON 格式" → 解析检查 |

### RLVR vs DPO

| | DPO | RLVR |
|--|-----|------|
| 奖励来源 | 人类/GPT-4o 偏好 | 程序化验证 |
| 适用任务 | 开放式（写作、对话） | 可验证任务（数学、代码） |
| 成本 | 需要构建偏好对 | 只需要标准答案 |
| 效果 | 泛化性好 | 在可验证任务上更精准 |

### 在 Tülu 3 中的角色

```
Base → SFT → DPO → RLVR → Final Model
                          ↑
                    仅用于数学/代码/指令遵循
                    进一步提升可验证任务的分数
```

> **对应 Tülu 3 论文 Section 4.3**：RLVR 在数学 (GSM8K) 上比纯 DPO 提升 ~5%，但在开放式任务上无明显增益。
> 这说明不同训练阶段解决不同问题：SFT→格式，DPO→偏好，RLVR→精确性。

### DeepSeek-R1 的成功

DeepSeek-R1 在 RLVR 上走得更远：用大规模思维链 RL 训练，让模型学会"思考过程"。
这也是 RLVR 思路的延伸——对有标准答案的任务，直接用正确性作为奖励信号。

## 概念 6：安全数据的"正交性"

### Tülu 3 的关键发现

在 SFT 数据中加入 10 万条安全数据（WildGuardMix + WildJailbreak），结果发现：

| 指标 | 无安全数据 | 有安全数据 | 变化 |
|------|----------|----------|------|
| MMLU | 65.2% | 65.1% | -0.1%（忽略不计） |
| GSM8K | 72.1% | 71.8% | -0.3%（忽略不计） |
| HumanEval | 61.5% | 61.2% | -0.3%（忽略不计） |
| Safety ASR↓ | 45% | 8% | **-37%（显著降低）** |

**结论：安全和能力是"正交"的——它们占用不同的参数空间。**

直觉理解：
- "知道法国首都是巴黎"和"知道不应该教人做炸弹"用的是模型的不同参数
- 学安全拒绝不会"挤掉"事实知识

### Over-refusal 问题

但安全数据**过多**会导致 **over-refusal**（过度拒绝）：

```
用户: "如何开锁？"（实际场景：被锁在门外）
Over-refusal 模型: "对不起，我不能帮助你进行非法活动。"
正常模型: "如果你被锁在门外，可以尝试联系锁匠..."
```

### CoCoNot 数据的作用

**CoCoNot（Contextual Counter-Not）** 数据专门对抗 over-refusal：
- 包含看似敏感但实际无害的问题
- 教模型区分"真正有害"和"表面敏感但实际良性"的请求

> **对应 Tülu 3 论文 Section 3.2**：加入 ~11K CoCoNot 数据后，over-refusal rate 从 25% 降到 8%，而 ASR 几乎不变。
> 这进一步证明安全能力的不同维度也是正交的。

## 概念 7：数据去污染

### 问题：训练数据泄漏

如果训练数据包含评估 benchmark 的原题，模型在评估时的高分就是"虚假"的——它只是"背过答案"。

**示例**：
```
训练数据中: "The capital of France is Paris. (MMLU Geography)"
MMLU 评估时: "What is the capital of France? A.Paris B.London C.Berlin D.Madrid"
→ 模型答对了，但不是因为"理解"，而是因为"背过"
```

### Tülu 3 的三种检测方法

| 方法 | 原理 | 优缺点 |
|------|------|--------|
| 全字符串匹配 | 归一化后完全相同 | 快速但过于严格 |
| N-gram 匹配 | 13-gram 重叠率 > 70% | 平衡速度和准确度 |
| Embedding 相似度 | 语义向量相似度 | 最准确但最慢 |

### 发现

Tülu 3 检查了所有公开 SFT 数据集，发现：
- 部分数据集与 MMLU 有 **>5% 重叠**
- 与 GSM8K 的重叠尤其严重（数学题容易重复）
- 去污染后，模型在这些 benchmark 上的分数下降 1-3%

### 本项目的做法

我们使用 13-gram 匹配（与 Tülu 3 一致），对子采样后的数据做去污染：
1. 提取训练数据和评估集的 13-gram
2. 计算 Jaccard 重叠率
3. 重叠率 > 70% 的样本标记为"被污染"
4. 从训练集中去除被污染样本

> **对应 Tülu 3 论文 Section 3.3**：去污染是确保评估结果可信的关键步骤。
> 不做去污染的实验结果不具备参考价值。

## 概念 8：消融实验

### 什么是消融实验？

**消融实验（Ablation Study）**：去掉一个组件，看效果变化多少。

就像医学上的"对照实验"——要证明某种药有效，就需要一组不吃药的对照组。

### 为什么必须做消融？

面试场景：
> 面试官："你的 pipeline 有 SFT、DPO、安全数据、CoCoNot 这些组件。你怎么知道每个都有用？"
> 
> 好回答："我做了消融实验。去掉安全数据后，ASR 从 15% 升到 65%，但 MMLU 不变，证明安全数据独立有效。去掉 CoCoNot 后，over-refusal 从 8% 升到 25%，证明它有效对抗过度拒绝。"
> 
> 差回答："我觉得每个都有用吧..."

### 本项目的 5 组消融

| 实验 | 去掉什么 | 预期影响 | 验证什么 |
|------|---------|---------|----------|
| Full | 无（baseline） | — | 对照基准 |
| -Safety | WildGuardMix + WildJailbreak | Safety ASR↑, 能力不变 | 安全正交性 |
| -Math | NuminaMath-TIR | 数学↓, 其他不变 | 数学数据贡献 |
| -CoCoNot | CoCoNot | Over-refusal↑ | 防过度拒绝 |
| -NoRobots | No Robots | 对话质量↓ | 人工数据价值 |
| -FLAN | FLAN v2 | 指令遵循↓ | 通用指令数据 |

> **对应 Tülu 3 论文 Section 5.3**：每个消融实验都用相同的训练配置，只改变数据组成。
> 这样的对比才能准确归因——效果变化一定来自数据变化，而非训练配置变化。

## 概念 9：DeepSeek 等模型也是这个链路

### Post-training 是通用范式

几乎所有现代 Chat Model 都遵循相同的 Post-training 链路：

```
DeepSeek:
  DeepSeek-V3-Base → SFT + RL → DeepSeek-V3 → 思维链 RL → DeepSeek-R1

Llama 3.1:
  Llama-3.1-8B (Base) → SFT → RLHF (DPO) → Llama-3.1-8B-Instruct

Qwen 2.5:
  Qwen2.5-1.5B (Base) → SFT → DPO → Qwen2.5-1.5B-Instruct

GPT-4:
  GPT-4-Base → SFT → RLHF → GPT-4 (ChatGPT)

Claude:
  Claude-Base → SFT → Constitutional AI (RLAIF) → Claude
```

### 每个阶段解决什么问题

| 阶段 | 解决的问题 | 类比 |
|------|----------|------|
| Pre-training | 学知识、学语言 | 上学读书 |
| SFT | 学对话格式、角色意识 | 入职培训 |
| DPO/RLHF | 学偏好、提升质量 | 导师指导 |
| RLVR | 精确任务优化 | 专项训练 |

### 为什么理解 Post-training 很重要？

1. **Pre-training 越来越标准化**：大家都用类似的数据和方法
2. **Post-training 是差异化竞争力**：同样的 Base Model，不同的 Post-training 产出截然不同的产品
3. **成本可控**：Pre-training 需要数千万美元，Post-training 只需数万美元
4. **迭代更快**：Pre-training 训练几个月，Post-training 几天就能迭代一版

> 可以说，**Pre-training 决定了模型的上限，Post-training 决定了模型的下限和用户体验**。
> 本项目就是帮你从零理解和实践 Post-training 的完整流程。

---

## 总结：Post-Training 知识图谱

```
Post-Training
├── SFT（监督微调）
│   ├── 数据：指令-回复对（57K）
│   ├── 方法：标准交叉熵损失
│   ├── 关键技巧：数据配比（技能隔离）
│   └── 效果：学会对话格式
│
├── DPO（直接偏好优化）
│   ├── 数据：偏好对（30K）
│   ├── 方法：Bradley-Terry + sigmoid loss
│   ├── 超参：β=5, lr=5e-7
│   └── 效果：提升回答质量和安全性
│
├── RLVR（可验证奖励 RL）[本项目不实现]
│   ├── 数据：有标准答案的任务
│   ├── 方法：程序化验证作为奖励
│   └── 效果：精确任务进一步提升
│
├── 安全
│   ├── 安全正交性：安全数据不损能力
│   ├── Over-refusal：过度安全的副作用
│   └── CoCoNot：对抗过度拒绝
│
├── 评估
│   ├── Benchmark：MMLU, HellaSwag, ARC, ...
│   ├── 安全：ASR + Over-refusal 双指标
│   └── 去污染：防止数据泄漏
│
└── 消融实验
    └── 量化每个组件的贡献
```

**下一步**：打开 Notebook 01，深入分析 SFT 和 DPO 的实际数据。